#Performance Sensitivity Analysis to 5 catchments characteristics

**Author:** Lionel Cedric Gohouede


In [ ]:
import pandas as pd
import numpy as np
from scipy import stats
from pathlib import Path
import warnings
warnings.filterwarnings('ignore')

from google.colab import drive
drive.mount("/content/drive")
DATA_DIR = Path("/content/drive/MyDrive/Colab Notebooks/Data")

MODEL_FILES = {
    'HyMoLAP': 'HyMoLAP_Simulation_Data_CAMELS_FR.csv',
    'dHyMoLAP': 'dHyMoLAP_Simulation_Data_CAMELS_FR.csv',
    'HBV':      'HBV_Simulation_Data_CAMELS_FR.csv',
}

PERF_BINS = [-np.inf, 0.50, 0.70, 0.80, 1.00]
PERF_LABELS = ['Unsatisfactory', 'Satisfactory', 'Good', 'Very Good']

ATTRIBUTES = [
    {'name': 'Rainfall', 'corr_col': 'rainfall_annual', 'cat_col': 'rainfall_tertile',
     'order': ['Low', 'Medium', 'High']},
    {'name': 'Q/P Ratio', 'corr_col': 'qp_ratio', 'cat_col': 'qp_category',
     'order': ['Wet', 'Balanced', 'Dry']},
    {'name': 'Snow Fraction', 'corr_col': 'snow_fraction', 'cat_col': 'snow_category',
     'order': ['Negligible', 'Low', 'Moderate', 'High']},
    {'name': 'Geographic Zone', 'corr_col': None, 'cat_col': 'geo_zone',
     'order': ['Alpine', 'Mediterranean', 'Atlantic', 'Continental']},
    {'name': 'Catchment Size', 'corr_col': 'catchment_area', 'cat_col': 'size_category',
     'order': ['Small', 'Medium', 'Large']},
]


# ---------------------------------------------------------------- Data loading

def standardize_id(df, old_col='sta_code_h3', new_col='station_id'):
    if old_col in df.columns:
        df.rename(columns={old_col: new_col}, inplace=True)
    return df


def load_soil(data_dir):
    """Pivot soil attributes (mean stat) by aggregation level into wide format."""
    soil_raw = pd.read_csv(data_dir / 'CAMELS_FR_soil_general_attributes.csv', sep=';')
    soil_mean = soil_raw[soil_raw['sol_stat'] == 'mean'].copy()

    parts = []
    for level in ['no', 'topsoil', 'top_subsoil']:
        subset = soil_mean[soil_mean['sol_agg_level'] == level].drop(['sol_stat', 'sol_agg_level'], axis=1)
        subset = subset.rename(columns={c: f'sol_{level}_{c}' for c in subset.columns if c != 'sta_code_h3'})
        parts.append(standardize_id(subset))

    soil = parts[0]
    for p in parts[1:]:
        soil = soil.merge(p, on='station_id', how='outer')
    return soil


def load_shared_attributes(data_dir):
    """Load and merge all model-independent catchment attribute datasets (loaded once)."""
    climate = pd.read_csv(data_dir / 'CAMELS_FR_climatic_statistics.csv', sep=';')
    hydro = pd.read_csv(data_dir / 'CAMELS_FR_hydrological_signatures.csv', sep=';')
    hydro_yearly = pd.read_csv(data_dir / 'CAMELS_FR_hydroclimatic_statistics_joint_availability_yearly.csv', sep=';')
    topo = pd.read_csv(data_dir / 'CAMELS_FR_topography_general_attributes.csv', sep=';')
    geology = pd.read_csv(data_dir / 'CAMELS_FR_geology_attributes.csv', sep=';')
    hydrogeology = pd.read_csv(data_dir / 'CAMELS_FR_hydrogeology_attributes.csv', sep=';')
    landcover = pd.read_csv(data_dir / 'CAMELS_FR_land_cover_attributes.csv', sep=';')
    station = pd.read_csv(data_dir / 'CAMELS_FR_station_general_attributes.csv', sep=';', on_bad_lines='skip')
    site = pd.read_csv(data_dir / 'CAMELS_FR_site_general_attributes.csv', sep=';', on_bad_lines='skip')
    nestedness = pd.read_csv(data_dir / 'CAMELS_FR_catchment_nestedness_information.csv', sep=';')
    dams = pd.read_csv(data_dir / 'CAMELS_FR_human_influences_dams.csv', sep=';')
    soil = load_soil(data_dir)

    frames = [climate, hydro, hydro_yearly, topo, soil, geology, hydrogeology, landcover, nestedness, dams, station]
    for df in frames:
        standardize_id(df)
    standardize_id(site, old_col='sit_code_h3')
    frames.append(site)

    shared = None
    for df in frames:
        if 'station_id' not in df.columns:
            continue
        if shared is None:
            shared = df.copy()
        else:
            shared = shared.merge(df, on='station_id', how='outer', suffixes=('', '_dup'))
            shared = shared.loc[:, ~shared.columns.str.endswith('_dup')]
    return shared


def engineer_features(data):
    """Derive only the attributes required for the sensitivity analysis."""
    if 'sta_area_snap' in data.columns:
        data['catchment_area'] = data['sta_area_snap']

    if 'cli_psol_frac_safran' in data.columns:
        data['snow_fraction'] = data['cli_psol_frac_safran']

    if 'cli_prec_mean_yr' in data.columns:
        data['rainfall_annual'] = data['cli_prec_mean_yr']
    elif 'cli_prec_mean' in data.columns:
        data['rainfall_annual'] = data['cli_prec_mean'] * 365.25

    if 'hyd_q_mean' in data.columns and 'cli_prec_mean' in data.columns:
        data['qp_ratio'] = (data['hyd_q_mean'] / (data['cli_prec_mean'] + 0.01)).clip(0, 1)
        data['catchment_type_wet'] = (data['qp_ratio'] > 0.5).astype(int)
        data['catchment_type_dry'] = (data['qp_ratio'] < 0.3).astype(int)

    if all(c in data.columns for c in ['sta_x_l93', 'sta_y_l93', 'top_altitude_mean']):
        x, y, alt = data['sta_x_l93'], data['sta_y_l93'], data['top_altitude_mean']
        data['zone_Alpine'] = ((alt > 800) & (x > 800000)).astype(int)
        data['zone_Mediterranean'] = ((y < 6200000) & (alt < 500)).astype(int)
        data['zone_Atlantic'] = ((x < 600000) & (data['zone_Alpine'] == 0)).astype(int)

    return data


def add_categories(data):
    """Build the categorical bins used in each attribute analysis."""
    if 'rainfall_annual' in data.columns:
        data['rainfall_tertile'] = pd.qcut(data['rainfall_annual'], q=3, labels=['Low', 'Medium', 'High'],
                                            duplicates='drop')

    if 'qp_ratio' in data.columns:
        data['qp_category'] = np.select(
            [data['catchment_type_wet'] == 1, data['catchment_type_dry'] == 1],
            ['Wet', 'Dry'], default='Balanced')

    if 'snow_fraction' in data.columns:
        data['snow_category'] = pd.cut(data['snow_fraction'], bins=[0, 0.02, 0.1, 0.5, 1.0],
                                        labels=['Negligible', 'Low', 'Moderate', 'High'], include_lowest=True)

    if 'zone_Alpine' in data.columns:
        data['geo_zone'] = np.select(
            [data['zone_Alpine'] == 1, data['zone_Mediterranean'] == 1, data['zone_Atlantic'] == 1],
            ['Alpine', 'Mediterranean', 'Atlantic'], default='Continental')

    if 'catchment_area' in data.columns:
        data['size_category'] = pd.cut(data['catchment_area'], bins=[0, 100, 1000, float('inf')],
                                        labels=['Small', 'Medium', 'Large'])

    return data


def build_dataset(params_path, shared):
    params = pd.read_csv(params_path)
    params['Performance'] = pd.cut(params['NSE_val'], bins=PERF_BINS, labels=PERF_LABELS, right=True)
    data = params[['station_id', 'NSE_val', 'Performance']].merge(shared, on='station_id', how='left')
    return add_categories(engineer_features(data))


# -------------------------------------------------------------------- Analysis

def correlation(data, col):
    valid = data[['NSE_val', col]].dropna()
    if len(valid) <= 30:
        return None
    r, p = stats.spearmanr(valid['NSE_val'], valid[col])
    return {'rho': r, 'p': p, 'n': len(valid)}


def category_summary(data, cat_col, order):
    rows = []
    for cat in order:
        subset = data[data[cat_col] == cat]
        if len(subset) == 0:
            continue
        rows.append({
            'category': cat,
            'n': len(subset),
            'mean_nse': subset['NSE_val'].mean(),
            'std_nse': subset['NSE_val'].std(),
            'vg_pct': 100 * (subset['NSE_val'] > 0.80).sum() / len(subset),
        })
    return rows


def anova(data, cat_col, order):
    groups = [data[data[cat_col] == cat]['NSE_val'].dropna() for cat in order if len(data[data[cat_col] == cat]) > 0]
    if len(groups) < 2:
        return None
    f, p = stats.f_oneway(*groups)
    return {'f': f, 'p': p}


def run_analysis(data):
    results = {}
    for attr in ATTRIBUTES:
        entry = {}
        if attr['corr_col'] and attr['corr_col'] in data.columns:
            entry['corr'] = correlation(data, attr['corr_col'])
        if attr['cat_col'] in data.columns:
            entry['categories'] = category_summary(data, attr['cat_col'], attr['order'])
            entry['anova'] = anova(data, attr['cat_col'], attr['order'])
        results[attr['name']] = entry
    return results


# ---------------------------------------------------------------------- Run all

shared = load_shared_attributes(DATA_DIR)

model_data, model_results = {}, {}
for model_name, filename in MODEL_FILES.items():
    data = build_dataset(DATA_DIR / filename, shared)
    model_data[model_name] = data
    model_results[model_name] = run_analysis(data)

# ------------------------------------------------------------------- Reporting

print("=" * 100)
print("MODEL COMPARISON: PERFORMANCE SENSITIVITY ANALYSIS")
print("=" * 100)

print("\nClass distribution:")
header = f"  {'Performance':16s}" + "".join(f"{m:>18s}" for m in MODEL_FILES)
print(header)
for label in PERF_LABELS:
    row = f"  {label:16s}"
    for model_name in MODEL_FILES:
        d = model_data[model_name]
        count = (d['Performance'] == label).sum()
        pct = 100 * count / len(d)
        row += f"{f'{pct:.1f}% ({count})':>18s}"
    print(row)

for attr in ATTRIBUTES:
    name = attr['name']
    print(f"\n{'=' * 100}\n{name.upper()}\n{'=' * 100}")

    if attr['corr_col']:
        print(f"\nSpearman correlation (NSE vs {attr['corr_col']}):")
        for model_name in MODEL_FILES:
            corr = model_results[model_name][name].get('corr')
            if corr:
                sig = "significant" if corr['p'] < 0.05 else "not significant"
                print(f"  {model_name:10s}: rho = {corr['rho']:+.3f}, p = {corr['p']:.3e} "
                      f"(n={corr['n']}) [{sig}]")
            else:
                print(f"  {model_name:10s}: insufficient data")

    print(f"\nPerformance by category (mean NSE ± std, n, %Very Good):")
    header = f"  {'Category':16s}" + "".join(f"{m:>30s}" for m in MODEL_FILES)
    print(header)
    for cat in attr['order']:
        row = f"  {cat:16s}"
        for model_name in MODEL_FILES:
            cats = {c['category']: c for c in model_results[model_name][name].get('categories', [])}
            if cat in cats:
                c = cats[cat]
                cell = f"{c['mean_nse']:.3f}±{c['std_nse']:.3f} (n={c['n']},VG={c['vg_pct']:.0f}%)"
            else:
                cell = "n/a"
            row += f"{cell:>30s}"
        print(row)

    print(f"\nANOVA (category effect on NSE):")
    for model_name in MODEL_FILES:
        av = model_results[model_name][name].get('anova')
        if av:
            sig = "significant" if av['p'] < 0.05 else "not significant"
            print(f"  {model_name:10s}: F = {av['f']:.2f}, p = {av['p']:.3e} [{sig}]")
        else:
            print(f"  {model_name:10s}: insufficient data")

Mounted at /content/drive
MODEL COMPARISON: PERFORMANCE SENSITIVITY ANALYSIS

Class distribution:
  Performance                HyMoLAP          dHyMoLAP               HBV
  Unsatisfactory         41.5% (228)        12.9% (71)        10.6% (58)
  Satisfactory           38.6% (212)       44.4% (244)       27.3% (150)
  Good                    15.8% (87)       30.4% (167)       30.4% (167)
  Very Good                4.0% (22)        12.2% (67)       31.7% (174)

RAINFALL

Spearman correlation (NSE vs rainfall_annual):
  HyMoLAP   : rho = +0.219, p = 2.299e-07 (n=549) [significant]
  dHyMoLAP  : rho = +0.352, p = 1.921e-17 (n=549) [significant]
  HBV       : rho = +0.205, p = 1.299e-06 (n=549) [significant]

Performance by category (mean NSE ± std, n, %Very Good):
  Category                               HyMoLAP                      dHyMoLAP                           HBV
  Low                  0.008±2.660 (n=183,VG=1%)     0.603±0.132 (n=183,VG=3%)    0.649±0.211 (n=183,VG=19%)
  Medium   